In [17]:
import torch
import imageio
from PINN import PINN
from Net import Net
import numpy as np
import pandas as pd
import os
import time
import sklearn.mixture as mixture
from unit_conversion import convert_wind
from scipy.interpolate import interp1d

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.set_default_tensor_type(torch.cuda.FloatTensor)

torch.set_printoptions(precision=8)

In [19]:
data_dir = f"{os.getcwd()}/pinn_test_data"
# load in data
df_wind_ch4 = pd.read_csv(data_dir + "/wind_ch4.csv")
df_true_emission = pd.read_csv(data_dir + "/selected_controll_release.csv")
source_points = np.load(data_dir + "/source_points.npy") # shape=(n_source, 3)
sensor_points = np.load(data_dir + "/sensor_points.npy") # shape=(n_sensor, 3)
#col_points = np.load(data_dir + "/col_points.npy")  # shape=(n_col, 3)
df_bounds = pd.read_csv(data_dir + "/bounds.csv", dtype='float32')
df_puff_simulation = pd.read_csv(data_dir + '/df_sim_puff_20220502008_0.csv')
x_min = df_bounds['x_min'][0]
x_max = df_bounds['x_max'][0]
y_min = df_bounds['y_min'][0]
y_max = df_bounds['y_max'][0]
z_min = df_bounds['z_min'][0]
z_max = df_bounds['z_max'][0]
active_source_idx = 3
# x_max = 1
# y_max = 1
# z_max = 1
tfinal = 5*60.
source_location = source_points

ws = df_puff_simulation['wind_speed.m/s'].to_numpy() # shape=(N_t,)
wd = df_puff_simulation['wind_direction'].to_numpy() # shape=(N_t,)
df_puff_simulation['x'], df_puff_simulation['y'] = convert_wind(ws,wd)

wind_function_x = interp1d(df_puff_simulation.index*60,df_puff_simulation.x)
wind_function_y = interp1d(df_puff_simulation.index*60,df_puff_simulation.y)

sensor_names = ['N','E','SE','S','SW','W','NW','C1','NE']
sensor_names = ['N','W','SW','S','SE','E','NE','C1','NW']

df_sensor = pd.DataFrame(sensor_points,columns = ['x','y','z'])
df_sensor['name'] = sensor_names
sensor_values_fn = dict()
for name in sensor_names:
    sensor_values_fn[name] = interp1d(df_puff_simulation.index*60,df_puff_simulation[name])
# ch4 = np.transpose(df_wind_ch4.iloc[:, 3:].to_numpy()) # shape=(N_obs, N_t)
sensor_names = df_wind_ch4.columns[3:]

In [20]:
df_sensor

,x,y,z,name
0,60.837350,75.084563,2.4,N
1,122.755778,31.351566,2.4,W
2,100.837025,2.203807,2.4,SW
3,68.353425,2.000000,2.4,S
4,32.244933,4.965663,2.4,SE
5,2.000000,39.602114,2.4,E
6,29.007035,75.790108,2.4,NE
7,45.327836,43.207236,2.4,C1
8,93.939519,76.053270,2.4,NW


In [21]:
sigma=.5
model = PINN([50,50,50,50])
source_vals = np.array([100/(60*5) if i ==active_source_idx else 0 for i in range(len(source_location))])
source_vals = np.zeros(5)+.1
print(source_vals)
model.set_location(source_location,[tfinal,x_max,y_max,z_max],source_values=source_vals,sigma=sigma,kappa=1e-2,trainable=True)


[0.1 0.1 0.1 0.1 0.1]


d:\andyh\Documents\Projects\mines\methane_project\methane_pinn_dev\Net.py:12: UserWarning: nn.init.xavier_uniform is now deprecated in favor of nn.init.xavier_uniform_.
  torch.nn.init.xavier_uniform(m.weight)
d:\andyh\Documents\Projects\mines\methane_project\methane_pinn_dev\Gaussian_Mixture.py:39: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.magnitude = nn.Parameter(torch.tensor(magnitude,requires_grad=True).float())


In [22]:
source_location

array([[61.84826691, 40.32822479,  4.5       ],
       [99.10094831, 54.69940709,  2.        ],
       [99.89962676, 24.72759871,  2.        ],
       [23.54499552, 57.03946784,  2.        ],
       [25.09781584, 22.62636785,  2.        ]])

In [23]:
optimizer = torch.optim.Adam(model.net.parameters(), lr=1e-2)
optimizer_q = torch.optim.Adam([model.source_mixture_hm.magnitude] , lr = 1e-3)
# from torch.optim.lr_scheduler import ExponentialLR
# scheduler = ExponentialLR(optimizer, gamma=0.999)  # Decay LR by 5% every epoch

In [ ]:
n= int(5e2)
# n=1
icn = int(1e2)
sn = int(5e3)
best_loss = np.inf
max_epochs = int(2e4)
print_freq = 100
sensor_time_points = 30
sampling_freq = 10 # how often to resample collocation and source points

for epoch in range(max_epochs):

    start_time = time.time()

    if epoch % sampling_freq == 0:

        source_collocation_points = model.source_points(sn,sigma*1) 
        ic_col = torch.cat([torch.zeros(icn,1), torch.rand(icn,1)*x_max, torch.rand(icn,1)*y_max, torch.rand(icn,1)*z_max], dim=1)
        collocation_points = torch.cat([torch.rand(n,1)*1.5*tfinal, torch.rand(n,1)*x_max*2 - x_max*.5, torch.rand(n,1)*y_max*2- y_max*.5, torch.rand(n,1)*z_max*2 - z_max*.5], dim=1)
        # collocation_points = torch.cat([collocation_points,ic_col,source_collocation_points])
        collocation_points = torch.cat([collocation_points,source_collocation_points,ic_col])
        collocation_points.requires_grad_(True)
        # t = np.floor(collocation_points[:,0:1].detach().numpy().flatten())
        # uv = torch.tensor([time_dict[t[i]] for i in range(len(t))])
        # uv = torch.ones(len(collocation_points),2)*10#wind tensor
        wind_tensor = torch.cat([torch.tensor(wind_function_x(collocation_points[:,0:1].detach().cpu().numpy())),torch.tensor(wind_function_y(collocation_points[:,0:1].detach().cpu().numpy()))],dim=1)
        # uv[:,1:]*= -1
        # print(uv)
        # wind_tensor = -1*torch.ones(len(collocation_points),2)
        # wind_tensor*= -1
    
    sensor_data_points = torch.empty(0,4).float()
    sensor_values = torch.empty( 0,1).float()
    for name in sensor_names:
        temp_time = torch.rand(sensor_time_points,1)*tfinal
        sensor_data_points = torch.cat([sensor_data_points,torch.cat([temp_time,torch.tensor(df_sensor[['x','y','z']][df_sensor.name==name].to_numpy()).repeat(sensor_time_points,1)],dim=1).float()])
        sensor_values = torch.cat([sensor_values,torch.tensor(sensor_values_fn[name](temp_time.detach().cpu().numpy())).float()])


    # optimizer_q.zero_grad()
    optimizer.zero_grad()

    loss_1 ,pde_1 = model.compute_pde_loss(collocation_points,wind_tensor) # PDE residual loss
    loss_2 = model.compute_negative_loss(collocation_points)
    loss_3 = model.compute_data_loss(sensor_data_points,sensor_values)
    loss_4 = torch.sum(torch.abs(model.source_mixture_hm.magnitude))
    loss_5 = model.compute_data_loss(torch.cat([torch.zeros(collocation_points.shape[0],1),collocation_points[:,1:]],dim=1),torch.zeros(collocation_points.shape[0],1))

    # loss = loss_3
    # loss = loss_1 + loss_3
    # loss = 5*loss_1 + loss_3/20 + loss_5
    # loss = loss_1 + loss_5
    loss = 100*loss_1+loss_2+loss_3/10+loss_5

    loss.backward()

    # compute norm of gradient of the network
    grad_norm = 0
    # for p in model.net.parameters():
    #     grad_norm += p.grad.data.norm(2).item()**2
    # grad_norm = grad_norm**0.5


    if loss.item() < best_loss:
        torch.save(model,'best_mod.m')
    
    optimizer.step()
    if epoch % 30 == 0:
        optimizer_q.step()
        optimizer_q.zero_grad()
        
    end_time = time.time()
    epoch_time = end_time - start_time
    # scheduler.step()

    if epoch % print_freq == 0:

        print('epoch: %d, loss: %1.3e, grad_norm: %1.3e, pde_res: %1.3e, time: %1.3e' % (epoch, loss.item(), grad_norm, loss_1.item(), epoch_time))
        print(model.source_mixture_hm.magnitude)
        # print(loss_1.item(),loss_2.item(),loss_3.item(),loss_4.item())



epoch: 0, loss: 2.583e+04, grad_norm: 0.000e+00, pde_res: 2.464e-01, time: 7.461e-01
Parameter containing:
tensor([0.10100000, 0.09900000, 0.10100000, 0.09900000, 0.10100000],
       requires_grad=True)
epoch: 100, loss: 1.818e+00, grad_norm: 0.000e+00, pde_res: 1.353e-02, time: 2.431e-02
Parameter containing:
tensor([0.10206740, 0.10024215, 0.10320961, 0.09663968, 0.10157195],
       requires_grad=True)
epoch: 200, loss: 9.997e-01, grad_norm: 0.000e+00, pde_res: 8.403e-03, time: 2.354e-02
Parameter containing:
tensor([0.10136725, 0.10157797, 0.10509434, 0.09380479, 0.10109120],
       requires_grad=True)
epoch: 300, loss: 7.958e-01, grad_norm: 0.000e+00, pde_res: 6.850e-03, time: 2.349e-02
Parameter containing:
tensor([0.09927898, 0.10303998, 0.10699997, 0.08994523, 0.10049568],
       requires_grad=True)
epoch: 400, loss: 6.042e-01, grad_norm: 0.000e+00, pde_res: 5.162e-03, time: 2.275e-02
Parameter containing:
tensor([0.09739634, 0.10385486, 0.10788143, 0.08717651, 0.09975281],
    

In [ ]:
model.source_mixture_hm.magnitude

Parameter containing:
tensor([ 0.19575909,  0.08334203, -0.00047329,  0.28731567, -0.00100133],
       requires_grad=True)

In [ ]:
model = torch.load('best_mod.m')

In [ ]:
import matplotlib.pyplot as plt
idx = torch.argsort(model.source_mixture_hm.magnitude)[-1]

# load the best model 
Z_value = source_location[idx,2]
# net.load_state_dict(torch.load('best_model.pth'))

# Define the grid and time steps
n= 100
x_grid = np.linspace(0, x_max, n)
y_grid = np.linspace(0, y_max, n)
z_grid = np.linspace(0, z_max, n)

X, Y, = np.meshgrid(x_grid, y_grid)
Z= X * 0 + Z_value

grid_points = np.vstack([X.ravel(), Y.ravel(), Z.ravel()]).T
print(np.vstack([X.ravel(), Y.ravel(), Z.ravel()]))
# grid_points = np.vstack([X.ravel(), Y.ravel(), Z.ravel()]).T  # Flatten the grid
grid_points = torch.tensor(grid_points, dtype=torch.float32)

time_steps = np.linspace(0, tfinal, 50)  # 10 time steps from 0 to 1



[[  0.           1.25252525   2.50505051 ... 121.49494949 122.74747475
  124.        ]
 [  0.           0.           0.         ...  78.          78.
   78.        ]
 [  2.           2.           2.         ...   2.           2.
    2.        ]]


In [ ]:
# save as a GIF
import imageio
import os
print(idx)
images = []
# source_loc = torch.tensor([[.5,.5,.5]])
source_loc = model.source_locs
for t in time_steps:
    t_tensor = torch.full((grid_points.shape[0], 1), t, dtype=torch.float32)  # Time input
    concentration = model.forward(torch.cat([t_tensor,grid_points],dim=1),scaled=True).cpu().detach().numpy().reshape(100, 100)  # Predict and reshape
    
    plt.figure()
    plt.contourf(X, Y, concentration, levels=50, cmap='viridis', vmin=0, vmax=3)  # Plot concentration as a contour plot
    plt.scatter(model.source_locs[:,0], model.source_locs[:,1], label='Source Locations')  # Plot the source location
    plt.scatter(sensor_points[:,0],sensor_points[:,1],marker='*')
    plt.scatter(model.source_locs[idx,0],model.source_locs[idx,1])
    plt.colorbar(label='Concentration')
    plt.title(f"Gas Concentration at t = {t:.2f}")
    plt.xlabel('x')
    plt.ylabel('y')
    plt.savefig(f"concentration_t_{t:.2f}.png")  # Save the plot as an image
    plt.close()
    
    images.append(imageio.imread(f"concentration_t_{t:.2f}.png"))  # Append the image to the list
    os.remove(f"concentration_t_{t:.2f}.png")  # Remove the image file

imageio.mimsave(f'output_gifs/test_site_example{int(time.time())}.gif', images)  # Save the images as a GIF



tensor(3)


C:\Users\andyh\AppData\Local\Temp\ipykernel_8568\2354341351.py:24: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  images.append(imageio.imread(f"concentration_t_{t:.2f}.png"))  # Append the image to the list


In [ ]:
print(model.source_mixture_hm.magnitude)

Parameter containing:
tensor([ 0.19570908,  0.08322579, -0.00047329,  0.28728750, -0.00100133],
       requires_grad=True)


In [ ]:
print(time.datetime())

AttributeError: module 'time' has no attribute 'datetime'

In [ ]:
model.compute_pde_loss(collocation_points,wind_tensor)

(tensor(703.2539, grad_fn=<MeanBackward0>),
 tensor(703.2539, grad_fn=<MeanBackward0>))

In [ ]:
print(model.source_mixture_hm.magnitude)
model.source_mixture_hm.magnitude.data += torch.tensor([0,0,0,1,0])

model.source_mixture_hm.magnitude.data *= 10
print(model.source_mixture_hm.magnitude)

Parameter containing:
tensor([-0.00094789, -0.00228275, -0.00227035, -0.00621119,  0.00703754],
       requires_grad=True)
Parameter containing:
tensor([-9.47888382e-03, -2.28275005e-02, -2.27034818e-02,  9.93788815e+00,
         7.03753829e-02], requires_grad=True)


In [ ]:
model.compute_pde_loss(collocation_points,wind_tensor)

(tensor(703.2539, grad_fn=<MeanBackward0>),
 tensor(703.2539, grad_fn=<MeanBackward0>))

In [ ]:
wind_tensor

tensor([[-2.87374417,  0.54449918],
        [-3.10724478,  0.31031961],
        [-2.83588419,  0.57199920],
        ...,
        [-2.61290716,  0.88899729],
        [-2.61290716,  0.88899729],
        [-2.61290716,  0.88899729]], dtype=torch.float64)

In [ ]:
model.t_max

300.0